# Job Description Extractor - Transformer NER Model

This notebook covers the design and evaluation of a Transformer-based Named Entity Recognition (NER) pipeline. It utilizes a pre-trained Token Classification model to parse raw Job Descriptions and extract technical stack items, organizational systems, and soft competencies.

In [1]:
import os
import torch
from transformers import AutoTokenizer, AutoModelForTokenClassification, pipeline
import pandas as pd
import numpy as np

C:\Users\Vansh Agrawal\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


C:\Users\Vansh Agrawal\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


## 1. Load Pre-trained Token Classification Model

We use a cased BERT/RoBERTa base model fine-tuned on CoNLL-03 English NER dataset to identify organizations, technologies (labeled as MISC or ORG), and locations.

In [2]:
model_name = "dslim/bert-base-NER"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForTokenClassification.from_pretrained(model_name)

# Create token classification pipeline
ner_pipeline = pipeline(
    "ner",
    model=model,
    tokenizer=tokenizer,
    aggregation_strategy="simple"
)
print("Transformer NER model loaded successfully!")

C:\Users\Vansh Agrawal\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Vansh Agrawal\.cache\huggingface\hub\models--dslim--bert-base-NER. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


Some weights of the model checkpoint at dslim/bert-base-NER were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


Device set to use cpu


Transformer NER model loaded successfully!


## 2. Test Skill Extraction on Job Descriptions

In [3]:
jd_example = """
We are looking for a Senior Software Engineer at Google in Bangalore. 
You will design high-throughput microservices using Python, FastAPI, and PostgreSQL.
Experience with Docker, Kubernetes, and AWS (Amazon Web Services) is a plus.
"""

entities = ner_pipeline(jd_example)
df_entities = pd.DataFrame(entities)
df_entities

,entity_group,score,word,start,end
0,ORG,0.998867,Google,50,56
1,LOC,0.998919,Bangalore,60,69
2,MISC,0.993850,Python,124,130
3,MISC,0.928708,FastAPI,132,139
4,MISC,0.902091,PostgreSQL,145,155
5,ORG,0.767985,Docker,173,179
6,ORG,0.696188,Kubernetes,181,191
7,ORG,0.578578,A,197,198
8,MISC,0.745874,##WS,198,200
9,ORG,0.967393,Amazon Web Services,202,221


## 3. Post-Processing to Extract Tech Stack Items

Since standard CoNLL categories are generic (LOC, ORG, PER, MISC), we use an auxiliary string-matching filter (or heuristics) to isolate programming tools and languages extracted via NER.

In [4]:
def extract_skills_from_ner(entities, text):
    extracted = []
    for ent in entities:
        # Extract entities tagged as MISC (Miscellaneous frameworks/tools) or ORG (Libraries/Platforms like PyTorch/Docker)
        if ent['entity_group'] in ('MISC', 'ORG'):
            extracted.append(ent['word'].strip())
    return list(set(extracted))

skills = extract_skills_from_ner(entities, jd_example)
print(f"Extracted Skills & Tech Stack: {skills}")

Extracted Skills & Tech Stack: ['Kubernetes', 'Amazon Web Services', 'A', 'Google', 'FastAPI', '##WS', 'PostgreSQL', 'Python', 'Docker']
